In [0]:
%sh
cd /Volumes/workspace/aero/flights
ls -lh flights.csv


In [0]:
files = ['flights', 'airports', 'airlines']

spark.sql("CREATE DATABASE IF NOT EXISTS aero")

for f in files:
    df = spark.read.csv(
        f'/Volumes/workspace/aero/flights/{f}.csv',
        header=True,
        inferSchema=True
    )

    df.write.mode("overwrite").saveAsTable(f'aero.{f}')



In [0]:
%sql
SHOW TABLES IN aero;


In [0]:
%sql
select * from aero.flights

In [0]:
%sql
select * from aero.airports

In [0]:
%sql

WITH avg_dep AS (
    SELECT 
        f.Origin AS AirportCode,
        a.AIRPORT AS AirportName,
        ROUND(AVG(f.DepDelay), 2) AS AvgDepDelay
    FROM aero.flights f
    JOIN aero.airports a
    ON f.Origin = a.IATA_CODE
    GROUP BY f.Origin, a.AIRPORT
),
smallest AS (
    SELECT *, 'Smallest Delay' AS DelayType
    FROM (
        SELECT *, ROW_NUMBER() OVER (ORDER BY AvgDepDelay ASC) AS rn
        FROM avg_dep
    ) t
    WHERE rn = 1
),
largest AS (
    SELECT *, 'Largest Delay' AS DelayType
    FROM (
        SELECT *, ROW_NUMBER() OVER (ORDER BY AvgDepDelay DESC) AS rn
        FROM avg_dep
    ) t
    WHERE rn = 1
)
SELECT AirportCode, AirportName, AvgDepDelay, DelayType
FROM smallest

UNION ALL

SELECT AirportCode, AirportName, AvgDepDelay, DelayType
FROM largest


In [0]:
%sql
select STATE as State, count(*) as NumAirports
from aero.airports
group by STATE
order by NumAirports desc

Databricks visualization. Run in Databricks to view.

In [0]:
%sql

SELECT 
    DATE_FORMAT(CAST(FlightDate AS DATE), 'yyyy-MM-dd') AS DATE,
    COUNT(*) AS NUM_OF_FLIGHTS,
    ROUND(MAX(ArrDelay), 2) AS MAX_ARRIVAL_DELAY,
    ROUND(AVG(ArrDelay), 2) AS AVG_ARRIVAL_DELAY
FROM aero.flights
GROUP BY DATE_FORMAT(CAST(FlightDate AS DATE), 'yyyy-MM-dd')
ORDER BY DATE



In [0]:
%sql
select a.Airline as AirlineName,
count(*) as NumFlights
from aero.flights f
join aero.airlines a
on f.UniqueCarrier = a.IATA_CODE
group by a.Airline
order by numflights desc

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW flights_per_carrier AS
SELECT
    a.AIRLINE AS AirlineName,
    COUNT(*) AS NumFlights
FROM aero.flights f
JOIN aero.airlines a
ON f.UniqueCarrier = a.IATA_CODE
GROUP BY a.AIRLINE


In [0]:
%sql

CREATE OR REPLACE TEMPORARY VIEW flights_per_carrier AS
SELECT
    a.AIRLINE AS AirlineName,
    COUNT(*) AS NumFlights
FROM aero.flights f
JOIN aero.airlines a
ON f.UniqueCarrier = a.IATA_CODE
GROUP BY a.AIRLINE




In [0]:
%sql
select (*) from flights_per_carrier limit 10

In [0]:
%sql
select (*) from flights_per_carrier

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW flights_top10_plus_other AS

SELECT f.AirlineName, f.NumFlights
FROM flights_per_carrier f
JOIN (
    SELECT AirlineName
    FROM flights_per_carrier
    ORDER BY NumFlights DESC
    LIMIT 10
) t
ON f.AirlineName = t.AirlineName

UNION ALL

SELECT 'OTHER' AS AirlineName, SUM(f.NumFlights) AS NumFlights
FROM flights_per_carrier f
LEFT JOIN (
    SELECT AirlineName
    FROM flights_per_carrier
    ORDER BY NumFlights DESC
    LIMIT 10
) t
ON f.AirlineName = t.AirlineName
WHERE t.AirlineName IS NULL




In [0]:
%sql
SELECT * 
FROM flights_top10_plus_other
LIMIT 20;

Databricks visualization. Run in Databricks to view.